# 🎨 Aesthetics Experiments — Meningioma Cohort

Side-by-side **visual style** exploration on the cleaned + imputed handoff from `output/datasets/`.

Run from **repo root** (`meningioma-atypier/`). Same cohort as the modelling notebook.

<details>
<summary><b>📦 What this notebook does</b></summary>

- Loads imputed parquet + builds `high_grade_label`
- Each library cell has its **own knobs** (different default variables) — edit those to try new graphs
- Split/colour is always **high-grade**
- Code cells start with commented **Best for / Weak at** notes

</details>


## 00 · Setup & data load

⚙️ Core imports + cohort + shared helpers (`lab`, `plot_frame`, palette).

**Per-cell knobs:** each graph cell starts with a `# ── knobs` block — change columns there (defaults differ by library).

<details>
<summary>🔧 How it works</summary>

- Prefers **MICE-imputed** parquet under `output/datasets/`
- Adds `high_grade_label` from `high_grade`
- Grouping stays **high-grade**; knobs pick demographic / radiological fields only

</details>


In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import HTML, SVG, display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)

OUTPUT_ROOT = Path("output")
DATASETS_DIR = OUTPUT_ROOT / "datasets"

DATASET_CANDIDATES = [
    DATASETS_DIR / "mice_imputed_df.parquet",
    DATASETS_DIR / "simple_imputed_df.parquet",
    DATASETS_DIR / "unimputed_df.parquet",
]
for _path in DATASET_CANDIDATES:
    if _path.exists():
        DATA_PATH = _path
        break
else:
    raise FileNotFoundError(f"No parquet found under {DATASETS_DIR}")

df = pd.read_parquet(DATA_PATH)

df = df.assign(
    high_grade_label=np.where(df["high_grade"], "High grade", "Low grade"),
)
GRADE_ORDER = ["Low grade", "High grade"]
PALETTE = {"Low grade": "#0072B2", "High grade": "#D55E00"}

# ── shared helpers (used by every plot cell) ────────────────────────────
# Each graph cell has its OWN knobs near the top — change those per chart.
# Suggested columns to paste into cell knobs:
#   Demographic:  "age", "sex", "age_bins_10"
#   Radiological: "adc_value", "tumor_volume", "max_diameter_cm", "edema_volume_cm3",
#                 "meningioma_count", "perifocal_edema", "mass_effect",
#                 "dwi_hyperintensity", "t2_hyperintensity"
COL_LABELS = {
    "age": "Age (years)",
    "sex": "Sex",
    "age_bins_10": "Age bin",
    "adc_value": "ADC value",
    "tumor_volume": "Tumor volume (cm³)",
    "max_diameter_cm": "Max diameter (cm)",
    "edema_volume_cm3": "Edema volume (cm³)",
    "meningioma_count": "Meningioma count",
    "perifocal_edema": "Perifocal edema",
    "mass_effect": "Mass effect",
    "dwi_hyperintensity": "DWI hyperintensity",
    "t2_hyperintensity": "T2 hyperintensity",
    "high_grade_label": "High grade",
}


def lab(col: str) -> str:
    return COL_LABELS.get(col, col.replace("_", " "))


def plot_frame(*cols: str) -> pd.DataFrame:
    use = [c for c in cols if c in df.columns]
    return df.dropna(subset=use).copy()


print(f"✅ Loaded {DATA_PATH.name}: {len(df)} rows × {len(df.columns)} cols")
print("🎛️ Edit the knobs block at the top of each plot cell (different defaults per library).")
df[["high_grade_label", "age", "adc_value", "tumor_volume", "max_diameter_cm", "edema_volume_cm3", "sex"]].head(3)


### 🧰 Visualization libraries

Gallery of stacks used in this notebook — all installed in your environment.

<details>
<summary><b>Library roster</b></summary>

| Library | Role |
|---------|------|
| **matplotlib** | Low-level canvas — full pixel control, journal export |
| **seaborn** | Statistical plots on matplotlib — distributions, heatmaps |
| **plotly** | Interactive HTML — hover, zoom, e-poster screens |
| **altair** | Declarative Vega-Lite — layered grammar, clean composition |
| **plotnine** | ggplot2 grammar in Python — faceting, density, boxplots |
| **bokeh** | Web-native interactivity — tooltips, pan/zoom |
| **pygal** | Glossy SVG output — lightweight vector graphics |
| **hvplot** | HoloViz one-liners — quick exploratory hexbins etc. |
| **lets-plot** | Kotlin ggplot engine — crisp box/violin aesthetics |
| **scienceplots** | Publication matplotlib styles — IEEE/Nature-like typography |

</details>


In [ ]:
def section_divider(emoji: str, title: str, body_html: str) -> None:
    """Render a horizontal rule + collapsible recipe block."""
    display(
        HTML(
            f"""
            <hr style="border:none;border-top:3px solid #ddd;margin:2rem 0 1rem;">
            <h3>{emoji} {title}</h3>
            <details>
              <summary><b>Recipe</b> — libs · snippet</summary>
              {body_html}
            </details>
            """
        )
    )


---

## 01 · Matplotlib — twin axis + inset

🎛️ `X_COL` × `Y_COL` by high-grade, cohort counts inset. Change knobs in setup to swap axes.

<details>
<summary>Recipe — <code>matplotlib</code></summary>

- **Pattern:** scatter + `twinx()` + `inset_axes()` driven by `X_COL` / `Y_COL` / `GROUP_COL`
- **Sell:** pixel-level layout for papers when seaborn's defaults get in the way

</details>


In [ ]:
# Available columns — copy/paste into the knobs in the next cell
df.columns.tolist()


In [ ]:
# Best for: pixel-perfect journal figures, twin axes, insets, custom annotations/artists.
# Weak at: quick statistical EDA (seaborn) and interactivity (plotly/altair/bokeh).
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ── knobs (this cell only) ──────────────────────────────────────────────
GROUP_COL = "high_grade_label"
X_COL = "age"            # demographic
Y_COL = "adc_value"      # radiological
# ────────────────────────────────────────────────────────────────────────

plot_df = plot_frame(X_COL, Y_COL, GROUP_COL)

fig, ax = plt.subplots(figsize=(8.5, 5), dpi=120)
for label in GRADE_ORDER:
    sub = plot_df.loc[plot_df[GROUP_COL] == label]
    ax.scatter(
        sub[X_COL],
        sub[Y_COL],
        s=36,
        alpha=0.65,
        c=PALETTE[label],
        edgecolors="white",
        linewidths=0.4,
        label=label,
        zorder=2,
    )
    ax.axhline(sub[Y_COL].mean(), color=PALETTE[label], ls="--", lw=1.2, alpha=0.85, zorder=1)

ax.set(
    xlabel=lab(X_COL),
    ylabel=lab(Y_COL),
    title=f"{lab(X_COL)} × {lab(Y_COL)} — means + cohort inset",
)
ax.set_ylim(bottom=0)
ax.legend(frameon=False, loc="lower left")

ax2 = ax.twinx()
means = [plot_df.loc[plot_df[GROUP_COL] == g, Y_COL].mean() for g in GRADE_ORDER]
ax2.set_ylim(ax.get_ylim())
ax2.set_ylabel(f"Mean {lab(Y_COL)} (dashed)")
ax2.set_yticks(means)
ax2.set_yticklabels([f"{g}\nμ={m:.2f}" for g, m in zip(GRADE_ORDER, means)], fontsize=8)

ins = inset_axes(ax, width="30%", height="28%", loc="upper right", borderpad=1.2)
counts = plot_df[GROUP_COL].value_counts().reindex(GRADE_ORDER)
ins.bar(GRADE_ORDER, counts.values, color=[PALETTE[g] for g in GRADE_ORDER], width=0.6)
ins.set_title("n", fontsize=9, pad=2)
ins.tick_params(labelsize=7)
ins.set_ylim(0, counts.max() * 1.25)
for i, v in enumerate(counts.values):
    ins.text(i, v + 2, str(int(v)), ha="center", va="bottom", fontsize=7)
ins.set_xticks(range(len(GRADE_ORDER)))
ins.set_xticklabels(["Low", "High"], fontsize=7)

fig.tight_layout()
plt.show()


---

## 02 · Seaborn — jointplot with marginals

🎻 `Y_COL` × `Z_COL` by high-grade + marginal KDEs (clipped to ≥ 0).

<details>
<summary>Recipe — <code>seaborn</code></summary>

- **Pattern:** `sns.jointplot(x=Y_COL, y=Z_COL, hue=GROUP_COL)` + clipped `kdeplot`
- Swap via setup knobs (`Y_COL` / `Z_COL`)

</details>


In [ ]:
# Available columns — copy/paste into the knobs in the next cell
df.columns.tolist()


In [ ]:
# Best for: fast statistical EDA — distributions, relationships, facets, heatmaps — with sane defaults.
# Weak at: exotic layouts / twin axes (matplotlib) and rich interactivity (plotly/altair).
# E-poster tip: fill=True on KDE; clip/cut so density cannot spill into impossible negatives.

# ── knobs (this cell only) ──────────────────────────────────────────────
GROUP_COL = "high_grade_label"
X_COL = "adc_value"          # radiological
Y_COL = "tumor_volume"       # radiological
# ────────────────────────────────────────────────────────────────────────

plot_df = plot_frame(X_COL, Y_COL, GROUP_COL)

x_clip = (0.0, float(plot_df[X_COL].max()))
y_clip = (0.0, float(plot_df[Y_COL].max()))

g = sns.jointplot(
    data=plot_df,
    x=X_COL,
    y=Y_COL,
    hue=GROUP_COL,
    hue_order=GRADE_ORDER,
    palette=PALETTE,
    kind="scatter",
    height=6,
    ratio=4,
    space=0.05,
    alpha=0.65,
    marginal_kws=dict(fill=True, alpha=0.35, common_norm=False, cut=0, clip=(0, None)),
)
kde_kw = dict(levels=4, thresh=0.05, clip=(x_clip, y_clip), cut=0)
g.plot_joint(sns.kdeplot, fill=True, alpha=0.18, **kde_kw)
g.plot_joint(sns.kdeplot, fill=False, alpha=0.95, linewidths=1.6, **kde_kw)
g.ax_joint.set_xlim(left=0)
g.ax_joint.set_ylim(bottom=0)
g.set_axis_labels(lab(X_COL), lab(Y_COL))
g.figure.suptitle(f"{lab(X_COL)} × {lab(Y_COL)} — denser fill = more patients", y=1.02)
g.ax_joint.legend(title=lab(GROUP_COL), frameon=False)
plt.show()


---

## 03 · Plotly — parallel coordinates

✨ Brush `FEATURE_COLS`, coloured by high-grade. Edit `FEATURE_COLS` in setup to try other variables.

<details>
<summary>Recipe — <code>plotly.express</code></summary>

- **Pattern:** `px.parallel_coordinates(..., dimensions=FEATURE_COLS, color=...)`

</details>


In [ ]:
# Available columns — copy/paste into the knobs in the next cell
df.columns.tolist()

In [ ]:
# Best for: interactive HTML explorers — hover, zoom, parallel coords, 3D — ready for slides/Dash.
# Weak at: publication typography polish (matplotlib/scienceplots) and declarative linked views (altair).
import plotly.express as px

# ── knobs (this cell only) ──────────────────────────────────────────────
GROUP_COL = "high_grade_label"
FEATURE_COLS = [
    "age",                 # demographic
    "max_diameter_cm",     # radiological
    "tumor_volume",        # radiological
    "edema_volume_cm3",    # radiological
    "adc_value",           # radiological
]
# ────────────────────────────────────────────────────────────────────────

dims = list(FEATURE_COLS)
plot_df = plot_frame(*dims, "high_grade", GROUP_COL)
plot_df["grade_num"] = plot_df["high_grade"].astype(int)

fig = px.parallel_coordinates(
    plot_df,
    dimensions=dims,
    color="grade_num",
    color_continuous_scale=[PALETTE["Low grade"], PALETTE["High grade"]],
    labels={c: lab(c) for c in dims} | {"grade_num": lab(GROUP_COL)},
    title=f"Brush axes to filter patients — features={dims}",
)
for i, dim in enumerate(dims):
    fig.data[0].dimensions[i].range = [0, float(plot_df[dim].max())]
fig.update_layout(coloraxis_colorbar=dict(title="High grade", tickvals=[0, 1], ticktext=["Low", "High"]))
fig.show()


---

## 04 · Altair — linked brush selection

📐 Brush `X_COL` × `Y_COL` → `Z_COL` histogram filters. All from setup knobs.

<details>
<summary>Recipe — <code>altair</code></summary>

- **Pattern:** `selection_interval()` + `transform_filter` on hist of `Z_COL`

</details>


In [ ]:
# Available columns — copy/paste into the knobs in the next cell
df.columns.tolist()


In [ ]:
# Best for: declarative linked views, brush/filter, facet/layer composition, JSON-serializable charts.
# Weak at: highly custom artist drawing (matplotlib) and massive datasets without aggregation.
import altair as alt

# ── knobs (this cell only) ──────────────────────────────────────────────
GROUP_COL = "high_grade_label"
X_COL = "age"                # demographic
Y_COL = "tumor_volume"       # radiological
Z_COL = "adc_value"          # radiological (linked hist)
# ────────────────────────────────────────────────────────────────────────

plot_df = plot_frame(X_COL, Y_COL, Z_COL, GROUP_COL)
brush = alt.selection_interval()
color = alt.Color(
    f"{GROUP_COL}:N",
    scale=alt.Scale(domain=GRADE_ORDER, range=list(PALETTE.values())),
    legend=alt.Legend(title=lab(GROUP_COL)),
)

points = (
    alt.Chart(plot_df)
    .mark_circle(opacity=0.75, size=70)
    .encode(
        x=alt.X(f"{X_COL}:Q", title=lab(X_COL)),
        y=alt.Y(f"{Y_COL}:Q", title=lab(Y_COL), scale=alt.Scale(domainMin=0)),
        color=alt.condition(brush, color, alt.value("#d0d0d0")),
        tooltip=[X_COL, Y_COL, Z_COL, GROUP_COL],
    )
    .add_params(brush)
    .properties(width=420, height=280, title=f"Brush a region of {lab(X_COL)} × {lab(Y_COL)}")
)

z_max = float(plot_df[Z_COL].max())
hist = (
    alt.Chart(plot_df)
    .mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3)
    .encode(
        x=alt.X(
            f"{Z_COL}:Q",
            bin=alt.Bin(maxbins=18, extent=[0, z_max]),
            title=lab(Z_COL),
            scale=alt.Scale(domainMin=0),
        ),
        y=alt.Y("count()", title="Patients"),
        color=alt.Color(
            f"{GROUP_COL}:N",
            scale=alt.Scale(domain=GRADE_ORDER, range=list(PALETTE.values())),
            legend=None,
        ),
    )
    .transform_filter(brush)
    .properties(width=420, height=160, title=f"{lab(Z_COL)} — selection only")
)

(
    (points & hist)
    .configure_axis(grid=False)
    .configure_view(strokeWidth=0)
)


---

## 05 · Plotnine — ggplot layers + facets

🦢 `X_COL` × `Y_COL` + LOESS by high-grade, faceted by `FACET_COL`.

<details>
<summary>Recipe — <code>plotnine</code></summary>

- **Pattern:** `ggplot(aes(X_COL, Y_COL, color=GROUP_COL)) + geom_smooth + facet_wrap(FACET_COL)`

</details>


In [ ]:
# Available columns — copy/paste into the knobs in the next cell
df.columns.tolist()


In [ ]:
# Best for: ggplot2-style layered grammar — facets, stats, themes — if you think in aes()+geom_*.
# Weak at: interactivity and non-grammar custom artists.
from plotnine import (
    aes,
    facet_wrap,
    geom_point,
    geom_smooth,
    ggplot,
    labs,
    scale_color_manual,
    scale_y_continuous,
    theme_minimal,
)

# ── knobs (this cell only) ──────────────────────────────────────────────
GROUP_COL = "high_grade_label"
X_COL = "age"                  # demographic
Y_COL = "edema_volume_cm3"     # radiological
FACET_COL = "sex"              # demographic
# ────────────────────────────────────────────────────────────────────────

plot_df = plot_frame(X_COL, Y_COL, GROUP_COL, FACET_COL)

(
    ggplot(plot_df, aes(x=X_COL, y=Y_COL, color=GROUP_COL))
    + geom_point(alpha=0.55, size=2)
    + geom_smooth(method="lowess", se=False, size=1.1)
    + scale_color_manual(values=[PALETTE[g] for g in GRADE_ORDER], breaks=GRADE_ORDER)
    + scale_y_continuous(limits=(0, None))
    + facet_wrap(f"~{FACET_COL}", nrow=1)
    + theme_minimal(base_size=11)
    + labs(
        title=f"{lab(X_COL)} × {lab(Y_COL)} with LOESS, facet={lab(FACET_COL)}",
        x=lab(X_COL),
        y=lab(Y_COL),
        color=lab(GROUP_COL),
    )
)


---

## 06 · Bokeh — linked brushing across plots

🌐 Select on `Y_COL` × `Z_COL` → linked `X_COL` × `Y_COL` (shared source). High-grade colours.

<details>
<summary>Recipe — <code>bokeh</code></summary>

- **Pattern:** shared `ColumnDataSource` + box/lasso select across two figures

</details>


In [ ]:
# Available columns — copy/paste into the knobs in the next cell
df.columns.tolist()


In [ ]:
# Best for: custom interactive web plots — linked brushing, JS callbacks, dashboards (Panel/Bokeh server).
# Weak at: one-liner EDA (hvplot/seaborn) and grammar-of-graphics layering (plotnine/altair).
from bokeh.io import output_notebook, show
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool, Range1d
from bokeh.plotting import figure
from bokeh.transform import factor_cmap

output_notebook(hide_banner=True)

# ── knobs (this cell only) ──────────────────────────────────────────────
GROUP_COL = "high_grade_label"
X_COL = "age"                  # demographic (bottom plot)
Y_COL = "adc_value"            # radiological (bottom y / shared)
Z_COL = "max_diameter_cm"      # radiological (top x)
W_COL = "tumor_volume"         # radiological (top y)
# ────────────────────────────────────────────────────────────────────────

plot_df = plot_frame(X_COL, Y_COL, Z_COL, W_COL, GROUP_COL)
src = ColumnDataSource(plot_df)

tooltips = [
    (lab(GROUP_COL), f"@{GROUP_COL}"),
    (lab(X_COL), f"@{X_COL}{{0.0}}"),
    (lab(Y_COL), f"@{Y_COL}{{0.00}}"),
    (lab(Z_COL), f"@{Z_COL}{{0.0}}"),
    (lab(W_COL), f"@{W_COL}{{0.0}}"),
]
if "patient_code" in plot_df.columns:
    tooltips = [("Patient", "@patient_code")] + tooltips

hover = HoverTool(tooltips=tooltips)
color = factor_cmap(GROUP_COL, palette=list(PALETTE.values()), factors=GRADE_ORDER)
tools = "box_select,lasso_select,pan,wheel_zoom,reset"

p_scatter = figure(
    width=640,
    height=300,
    title=f"Select on {lab(Z_COL)} × {lab(W_COL)} → bottom plot dims the rest",
    tools=[hover, tools],
    x_axis_label=lab(Z_COL),
    y_axis_label=lab(W_COL),
    x_range=Range1d(0, float(plot_df[Z_COL].max()) * 1.05),
    y_range=Range1d(0, float(plot_df[W_COL].max()) * 1.05),
)
p_scatter.scatter(
    Z_COL,
    W_COL,
    source=src,
    size=9,
    alpha=0.7,
    line_color="white",
    color=color,
    selection_color="#222222",
    nonselection_alpha=0.15,
)

p_linked = figure(
    width=640,
    height=180,
    title=f"Same selection on {lab(X_COL)} × {lab(Y_COL)}",
    tools="pan,wheel_zoom,reset",
    x_axis_label=lab(X_COL),
    y_axis_label=lab(Y_COL),
    y_range=Range1d(0, float(plot_df[Y_COL].max()) * 1.05),
)
p_linked.scatter(
    X_COL,
    Y_COL,
    source=src,
    size=8,
    alpha=0.7,
    line_color="white",
    color=color,
    selection_color="#222222",
    nonselection_alpha=0.12,
)

show(column(p_scatter, p_linked))


---

## 07 · Pygal — radar (SVG)

💎 Z-scored means of `FEATURE_COLS` by high-grade. Edit the feature list in setup.

<details>
<summary>Recipe — <code>pygal</code></summary>

- **Pattern:** `Radar` of group means on `FEATURE_COLS`

</details>


In [ ]:
# Available columns — copy/paste into the knobs in the next cell
df.columns.tolist()


In [ ]:
# Best for: standalone SVG exports — radar, treemap, gauge — pretty web/report embeds.
# Weak at: scientific scatter/stats depth and large interactive exploration.
from pygal import Radar
from pygal.style import Style

# ── knobs (this cell only) ──────────────────────────────────────────────
GROUP_COL = "high_grade_label"
FEATURE_COLS = [
    "age",                  # demographic
    "meningioma_count",     # radiological / burden
    "max_diameter_cm",      # radiological
    "tumor_volume",         # radiological
    "edema_volume_cm3",     # radiological
]
# ────────────────────────────────────────────────────────────────────────

features = list(FEATURE_COLS)
plot_df = plot_frame(*features, GROUP_COL)

z = plot_df[features].apply(lambda s: (s - s.mean()) / s.std(ddof=0))
means = (
    z.assign(**{GROUP_COL: plot_df[GROUP_COL]})
    .groupby(GROUP_COL)[features]
    .mean()
    .reindex(GRADE_ORDER)
)

custom_style = Style(
    colors=tuple(PALETTE[g] for g in GRADE_ORDER),
    background="#F7F7F7",
    plot_background="#FFFFFF",
    foreground="rgba(0, 0, 0, 0.87)",
    foreground_strong="#111111",
    foreground_subtle="rgba(0, 0, 0, 0.55)",
    font_family="Helvetica, Arial, sans-serif",
)
chart = Radar(
    title=f"Z-scored profile by {lab(GROUP_COL)} (pygal radar)",
    style=custom_style,
    fill=True,
    show_legend=True,
    human_readable=True,
    width=700,
    height=480,
)
chart.x_labels = [lab(c) for c in features]
for grade in GRADE_ORDER:
    chart.add(grade, means.loc[grade].tolist())

display(SVG(chart.render(is_unicode=True)))


---

## 08 · HvPlot — groupby widget explorer

⚡ `X_COL` × `Y_COL`, colour=`Z_COL`, widget by high-grade. Swap knobs in setup.

<details>
<summary>Recipe — <code>hvplot.pandas</code></summary>

- **Pattern:** `df.hvplot.scatter(..., groupby=GROUP_COL, dynamic=False)`

</details>


In [ ]:
# Available columns — copy/paste into the knobs in the next cell
df.columns.tolist()


In [ ]:
# Best for: interactive EDA one-liners on DataFrames (widgets/groupby/datashader) via HoloViz.
# Weak at: fine-grained grammar (altair/plotnine) and print-ready typography (scienceplots).
# Note: dynamic=False → HoloMap with a working widget (DynamicMap was empty in this hvplot build).
import holoviews as hv
import hvplot.pandas  # noqa: F401 — registers accessor

hv.extension("bokeh")

# ── knobs (this cell only) ──────────────────────────────────────────────
GROUP_COL = "high_grade_label"
X_COL = "age"                  # demographic
Y_COL = "edema_volume_cm3"     # radiological
Z_COL = "max_diameter_cm"      # radiological (point colour)
# ────────────────────────────────────────────────────────────────────────

plot_df = plot_frame(X_COL, Y_COL, Z_COL, GROUP_COL)

plot_df.hvplot.scatter(
    x=X_COL,
    y=Y_COL,
    c=Z_COL,
    cmap="Inferno",
    groupby=GROUP_COL,
    dynamic=False,
    hover_cols=[GROUP_COL, Z_COL],
    width=640,
    height=380,
    title=f"{lab(X_COL)} × {lab(Y_COL)} coloured by {lab(Z_COL)} — widget by {lab(GROUP_COL)}",
    colorbar=True,
    size=80,
    alpha=0.75,
    padding=0.05,
    xlim=(float(plot_df[X_COL].min()), float(plot_df[X_COL].max())),
    ylim=(0, float(plot_df[Y_COL].max()) * 1.05),
)


---

## 09 · Lets-Plot — interactive ggplot + 2D density

🧊 Density field on `X_COL` × `Y_COL`, points coloured by high-grade.

<details>
<summary>Recipe — <code>lets_plot</code></summary>

- **Pattern:** `geom_density2df` + `geom_point(color=GROUP_COL)`

</details>


In [ ]:
# Available columns — copy/paste into the knobs in the next cell
df.columns.tolist()


In [ ]:
# Best for: ggplot-like API with interactive HTML tooltips / polished defaults (Lets-Plot engine).
# Weak at: deep matplotlib customization and Python-native linked grammar (altair).
from lets_plot import (
    LetsPlot,
    aes,
    geom_density2df,
    geom_point,
    ggplot,
    ggsize,
    scale_color_manual,
    scale_x_continuous,
    scale_y_continuous,
    labs,
)

LetsPlot.setup_html(isolated_frame=True)

# ── knobs (this cell only) ──────────────────────────────────────────────
GROUP_COL = "high_grade_label"
X_COL = "age"                  # demographic
Y_COL = "max_diameter_cm"      # radiological
# ────────────────────────────────────────────────────────────────────────

plot_df = plot_frame(X_COL, Y_COL, GROUP_COL)
(
    ggplot(plot_df, aes(x=X_COL, y=Y_COL))
    + geom_density2df(alpha=0.55, show_legend=False)
    + geom_point(aes(color=GROUP_COL), size=2.5, alpha=0.7)
    + scale_color_manual(values=list(PALETTE.values()))
    + scale_x_continuous(limits=[float(plot_df[X_COL].min()), float(plot_df[X_COL].max())])
    + scale_y_continuous(limits=[0, float(plot_df[Y_COL].max())])
    + labs(
        title=f"{lab(X_COL)} × {lab(Y_COL)} density + points (lets-plot)",
        x=lab(X_COL),
        y=lab(Y_COL),
        color=lab(GROUP_COL),
    )
    + ggsize(640, 400)
)


---

## 10 · SciencePlots — Nature-style regression panel

🖋️ `X_COL` vs `Y_COL` and `X_COL` vs `Z_COL` by high-grade, SciencePlots styling.

<details>
<summary>Recipe — <code>scienceplots</code></summary>

- **Pattern:** two-panel OLS under `plt.style.context(["science", "no-latex"])`

</details>


In [ ]:
# Available columns — copy/paste into the knobs in the next cell
df.columns.tolist()


In [ ]:
# Best for: publication-ready matplotlib styling (Nature/IEEE/Science rcParams) with one context manager.
# Weak at: interactivity — it's a style pack, not a plotting grammar.
import scienceplots  # noqa: F401

# ── knobs (this cell only) ──────────────────────────────────────────────
GROUP_COL = "high_grade_label"
X_COL = "age"                  # demographic
Y_COL = "adc_value"            # radiological (panel 1)
Z_COL = "edema_volume_cm3"     # radiological (panel 2)
# ────────────────────────────────────────────────────────────────────────

plot_df = plot_frame(X_COL, Y_COL, Z_COL, GROUP_COL)

with plt.style.context(["ieee", "no-latex"]):
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.4), dpi=140, sharex=True)
    panels = [
        (Y_COL, lab(Y_COL), axes[0]),
        (Z_COL, lab(Z_COL), axes[1]),
    ]
    for ycol, ylabel, ax in panels:
        for label in GRADE_ORDER:
            sub = plot_df.loc[plot_df[GROUP_COL] == label]
            x = sub[X_COL].to_numpy(float)
            y = sub[ycol].to_numpy(float)
            ax.scatter(x, y, s=14, alpha=0.55, color=PALETTE[label], label=label)
            if len(sub) >= 3:
                coef = np.polyfit(x, y, 1)
                x_line = np.linspace(x.min(), x.max(), 50)
                y_line = np.clip(np.polyval(coef, x_line), 0, None)
                ax.plot(x_line, y_line, color=PALETTE[label], lw=1.4)
        ax.set(xlabel=lab(X_COL), ylabel=ylabel)
        ax.set_ylim(bottom=0)
        ax.legend(frameon=False, fontsize=7)
    fig.suptitle(f"{lab(X_COL)} relationships by {lab(GROUP_COL)} — SciencePlots", y=1.02)
    fig.tight_layout()
    plt.show()


---

## 🏁 Quick comparison

| # | Library | Default knobs (edit in cell) | Sell |
|---|---------|------------------------------|------|
| 01 | matplotlib | age × adc | twin axis + inset |
| 02 | seaborn | adc × tumor volume | jointplot + filled KDEs |
| 03 | plotly | age, Ø, volume, edema, adc | parallel coordinates |
| 04 | altair | age × volume → adc hist | linked brush |
| 05 | plotnine | age × edema · facet sex | LOESS + facet |
| 06 | bokeh | Ø × volume ↔ age × adc | linked selection |
| 07 | pygal | age, count, Ø, volume, edema | radar SVG |
| 08 | hvplot | age × edema · colour Ø | groupby widget |
| 09 | lets-plot | age × max diameter | 2D density + hover |
| 10 | scienceplots | age vs adc & edema | Nature-style 2-panel |

**E-poster pick:** seaborn/matplotlib (+ scienceplots) for static exports; plotly/altair for exploration first.
